<a href="https://colab.research.google.com/github/alpercagan/spectralbridge/blob/main/notebooks/02_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Week 2 Day 1: Feature Extraction - SpectralBridge Project
# ============================================================
# Extract 4 types of embeddings:
# 1. BEATs audio (self-supervised) - for SpectralBridge
# 2. DINOv2 image (self-supervised) - for SpectralBridge
# 3. CLAP audio (text-aligned) - for baseline comparison
# 4. CLIP image (text-aligned) - for baseline comparison

from google.colab import drive
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm
import torch

# Mount Drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/spectralbridge_data"

# Categories
SELECTED_CATEGORIES = [
    "playing piano", "playing acoustic guitar", "dog barking", "cat meowing",
    "helicopter", "typing on computer keyboard", "telephone bell ringing",
    "fire crackling", "ocean burbling", "thunder", "raining",
    "wind noise", "wind rustling leaves", "waterfall burbling",
    "hammering nails", "stream burbling",
]

# Verify data
audio_dir = Path(BASE_DIR) / "processed" / "audio"
image_dir = Path(BASE_DIR) / "processed" / "images"

total_audio = sum(len(list((audio_dir / cat).glob("*.wav")))
                 for cat in SELECTED_CATEGORIES if (audio_dir / cat).exists())
total_images = sum(len(list((image_dir / cat).glob("*.jpg")))
                  for cat in SELECTED_CATEGORIES if (image_dir / cat).exists())

print("="*60)
print("WEEK 2 DAY 1: FEATURE EXTRACTION")
print("="*60)
print(f"✓ Drive mounted: {BASE_DIR}")
print(f"✓ Categories: {len(SELECTED_CATEGORIES)}")
print(f"✓ Audio files: {total_audio}")
print(f"✓ Image files: {total_images}")
print("="*60)

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*60)

Mounted at /content/drive
WEEK 2 DAY 1: FEATURE EXTRACTION
✓ Drive mounted: /content/drive/MyDrive/spectralbridge_data
✓ Categories: 16
✓ Audio files: 1269
✓ Image files: 1269
✓ Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [2]:
# ============================================================
# Cell 2: Install Dependencies & Load Models
# ============================================================

print("Installing dependencies...")
print("="*60)

# Install required packages
!pip install -q timm transformers open_clip_torch torchaudio soundfile

print("✓ Packages installed")
print("="*60)
print("\nLoading models (this takes 2-3 minutes)...")
print("="*60)

import timm
import torch
import torchaudio
from transformers import ClapModel, ClapProcessor, CLIPModel, CLIPProcessor
from PIL import Image
import soundfile as sf

# 1. DINOv2 (Vision - Self-supervised)
print("Loading DINOv2...")
dinov2_model = timm.create_model(
    'vit_base_patch14_dinov2.lvd142m',
    pretrained=True,
    num_classes=0  # Remove classification head
).to(device).eval()
print("  ✓ DINOv2 loaded (768-d embeddings)")

# 2. CLIP (Vision - Text-aligned baseline)
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("  ✓ CLIP loaded (512-d embeddings)")

# 3. CLAP (Audio - Text-aligned baseline)
print("Loading CLAP...")
clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
print("  ✓ CLAP loaded (512-d embeddings)")

# 4. BEATs will be loaded separately (requires manual download)
print("\nBEATs: Will download checkpoint separately")
print("="*60)
print("✓ 3/4 models loaded successfully")
print("="*60)

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
✓ Packages installed

Loading models (this takes 2-3 minutes)...
Loading DINOv2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  ✓ DINOv2 loaded (768-d embeddings)
Loading CLIP...


config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

  ✓ CLAP loaded (512-d embeddings)

BEATs: Will download checkpoint separately
✓ 3/4 models loaded successfully


In [3]:
# ============================================================
# Cell 3: Create Feature Cache Directory
# ============================================================

features_dir = Path(BASE_DIR) / "features"
features_dir.mkdir(exist_ok=True)

print("="*60)
print("FEATURE CACHE SETUP")
print("="*60)
print(f"✓ Features will be saved to:")
print(f"  {features_dir}")
print()
print("Feature files to create:")
print("  1. beats_audio_features.npy      (BEATs, 1269 × 768)")
print("  2. dinov2_image_features.npy     (DINOv2, 1269 × 768)")
print("  3. clap_audio_features.npy       (CLAP, 1269 × 512)")
print("  4. clip_image_features.npy       (CLIP, 1269 × 512)")
print("  5. feature_metadata.npy          (filenames, categories)")
print()
print("Total size: ~15 MB (tiny!)")
print("="*60)

FEATURE CACHE SETUP
✓ Features will be saved to:
  /content/drive/MyDrive/spectralbridge_data/features

Feature files to create:
  1. beats_audio_features.npy      (BEATs, 1269 × 768)
  2. dinov2_image_features.npy     (DINOv2, 1269 × 768)
  3. clap_audio_features.npy       (CLAP, 1269 × 512)
  4. clip_image_features.npy       (CLIP, 1269 × 512)
  5. feature_metadata.npy          (filenames, categories)

Total size: ~15 MB (tiny!)


In [11]:
# ============================================================
# Cell 4: Load Wav2Vec2-BERT (Self-Supervised Audio Encoder)
# ============================================================

print("Loading Wav2Vec2-BERT (self-supervised audio encoder)...")
print("="*60)
print("Note: Using Wav2Vec2-BERT instead of BEATs")
print("Both are self-supervised, text-free audio encoders")
print("="*60)

from transformers import Wav2Vec2BertModel, AutoFeatureExtractor

# Load model and feature extractor (for audio preprocessing)
wav2vec_model = Wav2Vec2BertModel.from_pretrained(
    "facebook/w2v-bert-2.0"
).to(device).eval()

wav2vec_processor = AutoFeatureExtractor.from_pretrained(
    "facebook/w2v-bert-2.0"
)

print("✓ Wav2Vec2-BERT loaded (768-d embeddings)")
print("="*60)
print("✓ ALL 4 MODELS READY FOR FEATURE EXTRACTION")
print("="*60)
print()
print("Models loaded:")
print("  1. Wav2Vec2-BERT (audio, self-supervised) - 768-d")
print("  2. DINOv2 (vision, self-supervised) - 768-d")
print("  3. CLAP (audio baseline, text-aligned) - 512-d")
print("  4. CLIP (vision baseline, text-aligned) - 512-d")
print("="*60)

Loading Wav2Vec2-BERT (self-supervised audio encoder)...
Note: Using Wav2Vec2-BERT instead of BEATs
Both are self-supervised, text-free audio encoders


Loading weights:   0%|          | 0/773 [00:00<?, ?it/s]

✓ Wav2Vec2-BERT loaded (768-d embeddings)
✓ ALL 4 MODELS READY FOR FEATURE EXTRACTION

Models loaded:
  1. Wav2Vec2-BERT (audio, self-supervised) - 768-d
  2. DINOv2 (vision, self-supervised) - 768-d
  3. CLAP (audio baseline, text-aligned) - 512-d
  4. CLIP (vision baseline, text-aligned) - 512-d


In [21]:
# ============================================================
# Cell 5: Feature Extraction Helper Functions
# ============================================================

import librosa
import soundfile as sf
from PIL import Image
import numpy as np
from pathlib import Path

def load_audio(audio_path, target_sr=16000):
    """Load audio file and resample to target sample rate"""
    waveform, sr = sf.read(audio_path)
    if sr != target_sr:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
    return waveform

def extract_wav2vec_features(audio_path):
    """Extract Wav2Vec2-BERT features from audio"""
    waveform = load_audio(audio_path)

    # Preprocess
    inputs = wav2vec_processor(
        waveform,
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    # Extract features
    with torch.no_grad():
        outputs = wav2vec_model(**inputs)
        # Use mean of all time steps
        features = outputs.last_hidden_state.mean(dim=1).squeeze()

    return features.cpu().numpy()

def extract_dinov2_features(image_path):
    """Extract DINOv2 features from image"""
    img = Image.open(image_path).convert('RGB')

    # DINOv2 preprocessing
    transform = timm.data.create_transform(
        input_size=(518, 518),
        is_training=False,
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
    img_tensor = transform(img).unsqueeze(0).to(device)

    # Extract features
    with torch.no_grad():
        features = dinov2_model(img_tensor).squeeze()

    return features.cpu().numpy()

def extract_clap_features(audio_path):
    """Extract CLAP features from audio"""
    waveform = load_audio(audio_path, target_sr=48000)  # CLAP uses 48kHz

    # Preprocess
    inputs = clap_processor(
        audio=waveform,
        sampling_rate=48000,
        return_tensors="pt"
    ).to(device)

    # Extract features - handle different output types
    with torch.no_grad():
        output = clap_model.get_audio_features(**inputs)

        # Check if it's a tensor or wrapped object
        if isinstance(output, torch.Tensor):
            audio_features = output
        elif hasattr(output, 'pooler_output'):
            audio_features = output.pooler_output
        else:
            # Fallback: try indexing
            audio_features = output[0] if isinstance(output, (tuple, list)) else output

        audio_features = audio_features.squeeze()

    return audio_features.cpu().numpy()

def extract_clip_features(image_path):
    """Extract CLIP features from image"""
    img = Image.open(image_path).convert('RGB')

    # Preprocess
    inputs = clip_processor(images=img, return_tensors="pt").to(device)

    # Extract features - handle different output types
    with torch.no_grad():
        output = clip_model.get_image_features(**inputs)

        # Check if it's a tensor or wrapped object
        if isinstance(output, torch.Tensor):
            image_features = output
        elif hasattr(output, 'pooler_output'):
            image_features = output.pooler_output
        else:
            # Fallback: try indexing
            image_features = output[0] if isinstance(output, (tuple, list)) else output

        image_features = image_features.squeeze()

    return image_features.cpu().numpy()

print("="*60)
print("✓ Feature extraction functions defined")
print("="*60)

✓ Feature extraction functions defined


In [17]:
# ============================================================
# Cell 6: Extract Features from All Samples
# ============================================================

print("Starting feature extraction for 1,269 samples...")
print("Estimated time: 30-60 minutes on T4 GPU")
print("="*60)

# Prepare storage
wav2vec_features = []
dinov2_features = []
clap_features = []
clip_features = []
filenames = []
categories = []

# Track progress
total_samples = 0
failed_samples = []

# Loop through all categories
for category in tqdm(SELECTED_CATEGORIES, desc="Categories"):
    cat_audio_dir = audio_dir / category
    cat_image_dir = image_dir / category

    # Get all audio files in this category
    audio_files = sorted(cat_audio_dir.glob("*.wav"))

    for audio_file in tqdm(audio_files, desc=f"  {category}", leave=False):
        # Find corresponding image
        image_file = cat_image_dir / f"{audio_file.stem}.jpg"

        if not image_file.exists():
            print(f"⚠️  Missing image: {image_file.name}")
            failed_samples.append(str(audio_file))
            continue

        try:
            # Extract all 4 feature types
            wav2vec_feat = extract_wav2vec_features(audio_file)
            dinov2_feat = extract_dinov2_features(image_file)
            clap_feat = extract_clap_features(audio_file)
            clip_feat = extract_clip_features(image_file)

            # Store features
            wav2vec_features.append(wav2vec_feat)
            dinov2_features.append(dinov2_feat)
            clap_features.append(clap_feat)
            clip_features.append(clip_feat)

            # Store metadata
            filenames.append(audio_file.stem)
            categories.append(category)

            total_samples += 1

        except Exception as e:
            print(f"❌ Error processing {audio_file.name}: {e}")
            failed_samples.append(str(audio_file))
            continue

print("="*60)
print(f"✓ Extraction complete!")
print(f"  Successful: {total_samples}")
print(f"  Failed: {len(failed_samples)}")
print("="*60)

# Convert to numpy arrays
wav2vec_features = np.array(wav2vec_features)
dinov2_features = np.array(dinov2_features)
clap_features = np.array(clap_features)
clip_features = np.array(clip_features)

print(f"\nFeature shapes:")
print(f"  Wav2Vec2: {wav2vec_features.shape}")
print(f"  DINOv2:   {dinov2_features.shape}")
print(f"  CLAP:     {clap_features.shape}")
print(f"  CLIP:     {clip_features.shape}")
print("="*60)

Starting feature extraction for 1,269 samples...
Estimated time: 30-60 minutes on T4 GPU


Categories:   0%|          | 0/16 [00:00<?, ?it/s]

  playing piano:   0%|          | 0/80 [00:00<?, ?it/s]

  playing acoustic guitar:   0%|          | 0/80 [00:00<?, ?it/s]

  dog barking:   0%|          | 0/80 [00:00<?, ?it/s]

...

  wind rustling leaves:   0%|          | 0/80 [00:00<?, ?it/s]

  waterfall burbling:   0%|          | 0/80 [00:00<?, ?it/s]

  hammering nails:   0%|          | 0/80 [00:00<?, ?it/s]

  stream burbling:   0%|          | 0/80 [00:00<?, ?it/s]

✓ Extraction complete!
  Successful: 1269
  Failed: 0

Feature shapes:
  Wav2Vec2: (1269, 1024)
  DINOv2:   (1269, 768)
  CLAP:     (1269, 768, 2, 32)
  CLIP:     (1269, 50, 768)


In [22]:
# ============================================================
# Cell 6B: Re-extract CLAP and CLIP Features (Fixed)
# ============================================================

print("Re-extracting CLAP and CLIP features with correct pooling...")
print("This will take ~15 minutes")
print("="*60)

clap_features_fixed = []
clip_features_fixed = []

# Use the same order as before (filenames list)
for i, (filename, category) in enumerate(tqdm(zip(filenames, categories), total=len(filenames), desc="Re-extracting")):
    audio_file = audio_dir / category / f"{filename}.wav"
    image_file = image_dir / category / f"{filename}.jpg"

    try:
        # Re-extract CLAP and CLIP only
        clap_feat = extract_clap_features(audio_file)
        clip_feat = extract_clip_features(image_file)

        clap_features_fixed.append(clap_feat)
        clip_features_fixed.append(clip_feat)

    except Exception as e:
        print(f"❌ Error: {e}")
        # Use zeros as placeholder
        clap_features_fixed.append(np.zeros(512))
        clip_features_fixed.append(np.zeros(512))

# Convert to arrays
clap_features = np.array(clap_features_fixed)
clip_features = np.array(clip_features_fixed)

print("="*60)
print("✓ Re-extraction complete!")
print(f"\nCorrected shapes:")
print(f"  Wav2Vec2: {wav2vec_features.shape}")
print(f"  DINOv2:   {dinov2_features.shape}")
print(f"  CLAP:     {clap_features.shape}")
print(f"  CLIP:     {clip_features.shape}")
print("="*60)

Re-extracting CLAP and CLIP features with correct pooling...
This will take ~15 minutes


Re-extracting:   0%|          | 0/1269 [00:00<?, ?it/s]

✓ Re-extraction complete!

Corrected shapes:
  Wav2Vec2: (1269, 1024)
  DINOv2:   (1269, 768)
  CLAP:     (1269, 512)
  CLIP:     (1269, 512)


In [23]:
# ============================================================
# Cell 7: Save All Features to Google Drive
# ============================================================

print("Saving all features to Google Drive...")
print("="*60)

features_dir = Path(BASE_DIR) / "features"
features_dir.mkdir(exist_ok=True)

# Save feature arrays
np.save(features_dir / "wav2vec_audio_features.npy", wav2vec_features)
np.save(features_dir / "dinov2_image_features.npy", dinov2_features)
np.save(features_dir / "clap_audio_features.npy", clap_features)
np.save(features_dir / "clip_image_features.npy", clip_features)

print("✓ Feature arrays saved:")
print(f"  - wav2vec_audio_features.npy ({wav2vec_features.shape})")
print(f"  - dinov2_image_features.npy ({dinov2_features.shape})")
print(f"  - clap_audio_features.npy ({clap_features.shape})")
print(f"  - clip_image_features.npy ({clip_features.shape})")

# Save metadata
metadata = {
    'filenames': filenames,
    'categories': categories,
    'total_samples': len(filenames),
    'category_counts': {cat: categories.count(cat) for cat in SELECTED_CATEGORIES}
}

np.save(features_dir / "feature_metadata.npy", metadata)
print(f"\n✓ Metadata saved:")
print(f"  - feature_metadata.npy ({len(filenames)} samples)")

# Verify file sizes
print("\n" + "="*60)
print("Feature cache summary:")
print("="*60)
total_size_mb = 0
for file in features_dir.glob("*.npy"):
    size_mb = file.stat().st_size / (1024**2)
    total_size_mb += size_mb
    print(f"  {file.name}: {size_mb:.1f} MB")

print(f"\nTotal cache size: {total_size_mb:.1f} MB")
print("="*60)
print("✅ ALL FEATURES SAFELY CACHED TO DRIVE!")
print("="*60)
print("\n📌 IMPORTANT: These features are now permanent.")
print("   If Colab disconnects, you can reload them instantly!")
print("="*60)

Saving all features to Google Drive...
✓ Feature arrays saved:
  - wav2vec_audio_features.npy ((1269, 1024))
  - dinov2_image_features.npy ((1269, 768))
  - clap_audio_features.npy ((1269, 512))
  - clip_image_features.npy ((1269, 512))

✓ Metadata saved:
  - feature_metadata.npy (1269 samples)

Feature cache summary:
  wav2vec_audio_features.npy: 5.0 MB
  dinov2_image_features.npy: 3.7 MB
  clap_audio_features.npy: 2.5 MB
  clip_image_features.npy: 2.5 MB
  feature_metadata.npy: 0.0 MB

Total cache size: 13.7 MB
✅ ALL FEATURES SAFELY CACHED TO DRIVE!

📌 IMPORTANT: These features are now permanent.
   If Colab disconnects, you can reload them instantly!
